# account_userquestionrecord 테이블 전처리

In [1]:
import sys
from pathlib import Path
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

from src.utils.db import get_engine
engine = get_engine()

query = "SELECT * FROM accounts_userquestionrecord"
df = pd.read_sql(query, engine)

df.head()

,id,status,created_at,chosen_user_id,question_id,user_id,question_piece_id,has_read,answer_status,answer_updated_at,report_count,opened_times
0,771777,C,2023-04-28 12:27:49,849469,252,849436,998458,0,N,2023-04-28 12:27:49,0,0
1,771800,C,2023-04-28 12:28:02,849446,244,849436,998459,0,N,2023-04-28 12:28:02,0,0
2,771812,C,2023-04-28 12:28:09,849454,183,849436,998460,1,N,2023-04-28 12:28:09,0,0
3,771828,C,2023-04-28 12:28:16,847375,101,849436,998461,0,N,2023-04-28 12:28:16,0,0
4,771851,C,2023-04-28 12:28:26,849477,209,849436,998462,1,N,2023-04-28 12:28:26,0,0


### 중복 제거

In [3]:
duplicate_count = df.duplicated().sum()
print(duplicate_count)

0


중복값 X

### 결측치 확인

In [4]:
df.isnull().sum()

id                   0
status               0
created_at           0
chosen_user_id       0
question_id          0
user_id              0
question_piece_id    0
has_read             0
answer_status        0
answer_updated_at    0
report_count         0
opened_times         0
dtype: int64

결측 X

### 이상치 확인

In [5]:
df.describe()

,id,created_at,chosen_user_id,question_id,user_id,question_piece_id,has_read,answer_updated_at,report_count,opened_times
count,1.217558e+06,1217558,1.217558e+06,1.217558e+06,1.217558e+06,1.217558e+06,1.217558e+06,1217558,1.217558e+06,1.217558e+06
mean,5.957298e+07,2023-05-17 12:51:25.947103,1.092604e+06,6.843895e+02,1.105790e+06,7.413211e+07,5.551530e-01,2023-05-17 13:44:38.920359,1.765830e-04,6.332511e-02
min,7.717770e+05,2023-04-28 12:27:49,8.331120e+05,9.900000e+01,8.380230e+05,9.984580e+05,0.000000e+00,2023-04-28 12:27:49,0.000000e+00,0.000000e+00
25%,1.553965e+07,2023-05-10 01:42:11,8.836920e+05,2.750000e+02,8.846190e+05,1.854142e+07,0.000000e+00,2023-05-10 02:38:44.750000,0.000000e+00,0.000000e+00
50%,5.302680e+07,2023-05-15 15:43:01.500000,1.091749e+06,4.690000e+02,1.117537e+06,6.616843e+07,1.000000e+00,2023-05-15 16:29:08.500000,0.000000e+00,0.000000e+00
75%,9.480960e+07,2023-05-22 11:01:03.750000,1.235698e+06,9.420000e+02,1.259186e+06,1.176735e+08,1.000000e+00,2023-05-22 11:41:54.750000,0.000000e+00,0.000000e+00
max,1.616665e+08,2024-05-08 01:36:18,1.579422e+06,5.133000e+03,1.583358e+06,2.083515e+08,1.000000e+00,2024-05-08 01:36:18,1.400000e+01,3.000000e+00
std,4.608193e+07,NaN,2.027585e+05,6.250399e+02,2.062704e+05,5.757237e+07,4.969490e-01,NaN,2.040530e-02,3.011468e-01


In [10]:
print("숫자형 데이터 기초 통계 (마이너스 값이나 비정상적인 최대값 확인) ---")
print(df[['report_count', 'opened_times']].describe())
print("\n")

print("상태(status) 종류별 개수 확인 ---")
print(df['status'].value_counts())
print("\n")

print("답장 상태(answer_status) 종류별 개수 확인 ---")
print(df['answer_status'].value_counts())

숫자형 데이터 기초 통계 (마이너스 값이나 비정상적인 최대값 확인) ---
       report_count  opened_times
count  1.217558e+06  1.217558e+06
mean   1.765830e-04  6.332511e-02
std    2.040530e-02  3.011468e-01
min    0.000000e+00  0.000000e+00
25%    0.000000e+00  0.000000e+00
50%    0.000000e+00  0.000000e+00
75%    0.000000e+00  0.000000e+00
max    1.400000e+01  3.000000e+00


상태(status) 종류별 개수 확인 ---
status
C    1156322
I      60578
B        658
Name: count, dtype: int64


답장 상태(answer_status) 종류별 개수 확인 ---
answer_status
N    1097932
A     111761
P       7865
Name: count, dtype: int64


문자열도 3개씩 정리되어있음
이상치로 보이는 것은 없음

### 데이터 타입 변환

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1217558 entries, 0 to 1217557
Data columns (total 12 columns):
 #   Column             Non-Null Count    Dtype         
---  ------             --------------    -----         
 0   id                 1217558 non-null  int64         
 1   status             1217558 non-null  str           
 2   created_at         1217558 non-null  datetime64[us]
 3   chosen_user_id     1217558 non-null  int64         
 4   question_id        1217558 non-null  int64         
 5   user_id            1217558 non-null  int64         
 6   question_piece_id  1217558 non-null  int64         
 7   has_read           1217558 non-null  int64         
 8   answer_status      1217558 non-null  str           
 9   answer_updated_at  1217558 non-null  datetime64[us]
 10  report_count       1217558 non-null  int64         
 11  opened_times       1217558 non-null  int64         
dtypes: datetime64[us](2), int64(8), str(2)
memory usage: 111.5 MB


변환이 필요해보이는 컬럼은 보이지 않음

### 의문점

In [10]:
total_rows = len(df)
same_time_count = len(df[df['created_at'] == df['answer_updated_at']])
diff_count = total_rows - same_time_count

print(f"총 행 수: {total_rows:,}")
print(f"두 시간이 동일한 건수: {same_time_count:,}")
print(f"차이: {diff_count:,}\n")
print("-" * 50)

총 행 수: 1,217,558
두 시간이 동일한 건수: 1,096,506
차이: 121,052

--------------------------------------------------


#### 가설: answer_status가 N일 때 created_at = answer_updated_at의 조건일 것이다.

In [18]:
# 1. answer_status가 'N'인 데이터만 모두 가져오기
df_n_total = df[df['answer_status'] == 'N']
n_total_count = len(df_n_total)

print(f"▶ answer_status가 'N'인 전체 데이터 수: {n_total_count:,}건")

# 2. 'N' 중에서 시간이 같은 건수와 다른 건수 계산
n_same_count = len(df_n_total[df_n_total['created_at'] == df_n_total['answer_updated_at']])
n_diff_count = len(df_n_total[df_n_total['created_at'] != df_n_total['answer_updated_at']])

print(f"  ┣ 두 시간이 동일한 건수: {n_same_count:,}건")
print(f"  ┗ 두 시간이 다른 건수: {n_diff_count:,}건")

▶ answer_status가 'N'인 전체 데이터 수: 1,097,932건
  ┣ 두 시간이 동일한 건수: 1,096,506건
  ┗ 두 시간이 다른 건수: 1,426건


1,096,506건 + 1,426건 =  1,097,932건
- 전산 오류 예상
- 확인 작업 진행

In [19]:
condition = (df['answer_status'] == 'N') & (df['created_at'] != df['answer_updated_at'])
df_n_diff = df[condition].copy()

df_n_diff['time_diff'] = df_n_diff['created_at'] - df_n_diff['answer_updated_at']

display(df_n_diff[['id', 'answer_status', 'created_at', 'answer_updated_at', 'time_diff']].head())

,id,answer_status,created_at,answer_updated_at,time_diff
144062,3919269,N,2023-05-05 20:52:02,2023-05-05 20:52:01,0 days 00:00:01
144474,3924182,N,2023-05-05 22:35:44,2023-05-05 22:35:43,0 days 00:00:01
145282,3932822,N,2023-05-05 23:37:05,2023-05-05 23:37:04,0 days 00:00:01
145734,3937802,N,2023-05-05 23:58:19,2023-05-05 23:58:18,0 days 00:00:01
145908,3939441,N,2023-05-06 00:04:23,2023-05-06 00:04:22,0 days 00:00:01


- 시간 차이가 1초씩 차이남
    - 전산 오류가 맞는 것 같음
        - 질문 생성 시간보다 답변 업데이트 시간이 빠른 것은 오류임
- 1초 차이 나는 것을 위와 동기화 작업 진행

In [21]:
condition = (df['answer_status'] == 'N') & (df['created_at'] != df['answer_updated_at'])

df.loc[condition, 'answer_updated_at'] = df.loc[condition, 'answer_updated_at'] + pd.Timedelta(seconds=1)

remaining_diff = len(df[(df['answer_status'] == 'N') & (df['created_at'] != df['answer_updated_at'])])

print(f"시간이 다른 데이터 건수: {remaining_diff}건")

시간이 다른 데이터 건수: 0건


In [ ]:
# 1. answer_status가 'N'이 아닌 데이터 필터링
df_not_n = df[df['answer_status'] != 'N'].copy()

# 2. 'N'이 아닌 상태값들의 종류와 각각의 개수 확인
print(df_not_n['answer_status'].value_counts())
print("-" * 50)

# 3. 답변이 완료된 것들이니, 답장하는 데 걸린 시간(time_diff) 계산
df_not_n['time_diff'] = df_not_n['answer_updated_at'] - df_not_n['created_at']

# 4. 표 출력 (답장 걸린 시간 포함)
display(df_not_n[['id', 'answer_status', 'created_at', 'answer_updated_at', 'time_diff']].head())

▶ 'N'이 아닌 상태값 분포:
answer_status
A    111761
P      7865
Name: count, dtype: int64
--------------------------------------------------


,id,answer_status,created_at,answer_updated_at,time_diff
345,778243,P,2023-04-28 13:14:34,2023-05-06 16:49:33,8 days 03:34:59
1848,802848,P,2023-04-28 16:05:09,2023-05-06 16:49:21,8 days 00:44:12
3097,815819,P,2023-04-28 23:31:19,2023-05-06 10:09:27,7 days 10:38:08
10109,945319,P,2023-04-29 13:22:05,2023-05-06 10:31:30,6 days 21:09:25
11512,989867,A,2023-04-29 15:20:38,2023-05-12 15:33:47,13 days 00:13:09


중요한건 아니지만 answer_updated_at이 애매해서 조금 알아봄
- 1.5.0 업데이트 이전
    - 투표 받은 사람이 투표 해준사람에게 "뽑아줘서 고맙다" 라는 답장 가능
- 1.5.0 업데이트
    - 답장하기 기능에 10하트를 소모해야 답장을 보낼 수 있음
- 1.6.0 업데이트 
    - 채팅 기능 추가 200하트 소모


답장 시간
- 답장시 로그 업데이트처럼 해당 시간으로 업데이트가 되는 형식
- 답장을 안할 시에 질문 생성 시간에과 동일한 형태를 가짐


핵클 프로퍼티 테이블과 연동

- 앱 버전 이름도 있어서 이걸로 참고해서 보면 더 정확한 분석이 가능할듯!

## 타입 변환

In [23]:
id_cols = ['id', 'chosen_user_id', 'question_id', 'user_id', 'question_piece_id']

# object 타입으로 변경
df[id_cols] = df[id_cols].astype(object)